In [2]:
# =============================================================================
# TAREA 1: Exploración y Análisis de Calidad de Datos
# Curso: Ingeniería de Datos con Python
# =============================================================================

import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

# Crear carpetas necesarias
os.makedirs('homeworks/output', exist_ok=True)
os.makedirs('homeworks/tarea_1', exist_ok=True)


In [14]:
# === CARGAR DATOS ===
# Nota: El archivo está en la carpeta 'data' al mismo nivel que 'tarea_1'
# Estructura:
#   homeworks/
#   ├── data/
#   │   └── transacciones_raw.csv
#   └── tarea_1/
#       └── Tarea1_Exploracion.ipynb  ← Estás aquí

csv_path = '../data/transacciones_raw.csv'

if not os.path.exists(csv_path):
    print(f"\n⚠️ No se encuentra el archivo en: {csv_path}")
    print(f"Directorio actual: {os.getcwd()}")
    print("Verificando estructura de carpetas...")
    
    # Listar directorios
    print("\nContenido de homeworks:")
    if os.path.exists('..'):
        for item in os.listdir('..'):
            print(f"  - {item}")
    
    raise FileNotFoundError(f"No se puede encontrar {csv_path}")

# Cargar dataset
df = pd.read_csv(csv_path)
print(f"\n📁 Archivo cargado: {csv_path}")
print(f"📊 Shape: {df.shape[0]} filas × {df.shape[1]} columnas")
print("\n🔍 Primeros registros:")
print(df.head())


📁 Archivo cargado: ../data/transacciones_raw.csv
📊 Shape: 105 filas × 6 columnas

🔍 Primeros registros:
  transaction_id        date  customer_id   amount     status         store
0      TXN-00001  2023-03-02       1038.0  -194.07        NaN           NaN
1      TXN-00002  2023-04-06       1002.0   452.64  CANCELADA  Tienda_Norte
2      TXN-00003  2023-08-24       1017.0   344.76    FALLIDA    Tienda_Sur
3      TXN-00004  2023-01-20       1041.0   $85.16        NaN    Tienda_Sur
4      TXN-00005  02-07-2023       1000.0   $72.53    FALLIDA  Tienda_Norte


In [15]:
# =============================================================================
# 2. ANÁLISIS GENERAL
# =============================================================================
print("\n" + "=" * 70)
print("2. ANÁLISIS GENERAL DEL DATASET")
print("=" * 70)

print("\n📋 Columnas disponibles:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i}. {col}")

print("\n🔍 Tipos de datos:")
print(df.dtypes)

#Regresa las estadisticas del data_set
print("\n📊 Estadísticas básicas:")
print(df.describe(include='all'))


2. ANÁLISIS GENERAL DEL DATASET

📋 Columnas disponibles:
   1. transaction_id
   2. date
   3. customer_id
   4. amount
   5. status
   6. store

🔍 Tipos de datos:
transaction_id     object
date               object
customer_id       float64
amount             object
status             object
store              object
dtype: object

📊 Estadísticas básicas:
       transaction_id        date  customer_id amount     status       store
count             105         105    98.000000     98         80          86
unique            100          95          NaN     93          4           4
top         TXN-00039  2023-03-02          NaN  262.5  CANCELADA  Tienda_Sur
freq                2           2          NaN      2         22          29
mean              NaN         NaN  1021.795918    NaN        NaN         NaN
std               NaN         NaN    14.750823    NaN        NaN         NaN
min               NaN         NaN  1000.000000    NaN        NaN         NaN
25%               NaN   

In [17]:
# =============================================================================
# 3. ANÁLISIS DE VALORES NULOS
# =============================================================================
print("\n" +"=" * 70)
print("3. ANÁLISIS DE VALORES NULOS")
print("=" * 70)

nulos = df.isnull().sum()
porcentaje_nulos = (df.isnull().sum() / len(df)) * 100
print("\n❓ Valores nulos por columna:")
nulos_df = pd.DataFrame({
    'Columna': nulos.index,
    'Cantidad_Nulos': nulos.values,
    'Porcentaje': porcentaje_nulos.values
})
print(nulos_df.to_string(index=False))


3. ANÁLISIS DE VALORES NULOS

❓ Valores nulos por columna:
       Columna  Cantidad_Nulos  Porcentaje
transaction_id               0    0.000000
          date               0    0.000000
   customer_id               7    6.666667
        amount               7    6.666667
        status              25   23.809524
         store              19   18.095238


In [21]:
# =============================================================================
# 4. ANÁLISIS DE REGISTROS DUPLICADOS
# =============================================================================
print("\n" + "=" * 70)
print("4. ANÁLISIS DE DUPLICADOS")
print("=" * 70)

#total de duplicados
duplicados=df.duplicated().sum()
#total de duplicados por id
duplicados_por_id = df['transaction_id'].duplicated().sum()

print(f"\n🔄 Registros duplicados (todas las columnas): {duplicados}")
print(f"🔄 Transaction_id duplicados: {duplicados_por_id}")

if duplicados_por_id > 0:
    print("\n⚠️ Transaction_ids duplicados:")
    ids_duplicados = df[df['transaction_id'].duplicated(keep=False)]['transaction_id'].unique()
    for id_dup in ids_duplicados[:duplicados]:  # Mostrar todos los
        print(f"   • {id_dup}")


4. ANÁLISIS DE DUPLICADOS

🔄 Registros duplicados (todas las columnas): 5
🔄 Transaction_id duplicados: 5

⚠️ Transaction_ids duplicados:
   • TXN-00031
   • TXN-00036
   • TXN-00039
   • TXN-00043
   • TXN-00077


In [25]:
# =============================================================================
# 5. ANÁLISIS DE COLUMNA 'amount' (MONTOS)
# =============================================================================
print("\n" + "=" * 70)
print("5. ANÁLISIS DE MONTOS (amount)")
print("=" * 70)
# Tipos de datos en amount
tipos_amount = df['amount'].apply(type).value_counts()
for tipo, count in tipos_amount.items():
    print(f"   • {tipo.__name__}: {count}")
# Problemas específicos
montos_string = df['amount'].astype(str).str.contains('\$', na=False).sum()
montos_negativos = 0
montos_nulos = df['amount'].isnull().sum()

# Convertir a numérico para detectar negativos
amount_numeric = pd.to_numeric(df['amount'], errors='coerce')
montos_negativos = (amount_numeric < 0).sum()

print(f"\n⚠️ Problemas detectados en amount:")
print(f"   • Montos con formato string (con $): {montos_string}")
print(f"   • Montos negativos: {montos_negativos}")
print(f"   • Montos nulos: {montos_nulos}")

# Mostrar ejemplos
if montos_string > 0:
    print(f"\n📝 Ejemplos de montos que traen $ y se detectan como strings:")
    ejemplos = df[df['amount'].astype(str).str.contains('\$', na=False)]['amount'].head(3)
    for m in ejemplos:
        print(f"   • {m}")



5. ANÁLISIS DE MONTOS (amount)
   • str: 98
   • float: 7

⚠️ Problemas detectados en amount:
   • Montos con formato string (con $): 17
   • Montos negativos: 5
   • Montos nulos: 7

📝 Ejemplos de montos con problema:
   • $85.16
   • $72.53
   • $127.87


In [26]:
# =============================================================================
# 6. ANÁLISIS DE COLUMNA 'date' (FECHAS)
# =============================================================================
print("\n" + "=" * 70)
print("6. ANÁLISIS DE FECHAS (date)")
print("=" * 70)

# Identificar formatos de fecha
fechas_str = df['date'].astype(str)
formato_iso = fechas_str.str.match(r'\d{4}-\d{2}-\d{2}').sum()
formato_ddmmyyyy = fechas_str.str.match(r'\d{2}-\d{2}-\d{4}').sum()
fechas_nulas = df['date'].isnull().sum()

print(f"\n📅 Formatos de fecha detectados:")
print(f"   • ISO (YYYY-MM-DD): {formato_iso}")
print(f"   • DD-MM-YYYY: {formato_ddmmyyyy}")
print(f"   • Nulas: {fechas_nulas}")

# Mostrar ejemplos de fechas no estándar
if formato_ddmmyyyy > 0:
    print(f"\n📝 Ejemplos de fechas en formato DD-MM-YYYY:")
    ejemplos_fechas = df[~df['date'].astype(str).str.match(r'\d{4}-\d{2}-\d{2}', na=False)]['date'].head(3)
    for f in ejemplos_fechas:
        print(f"   • {f}")


6. ANÁLISIS DE FECHAS (date)

📅 Formatos de fecha detectados:
   • ISO (YYYY-MM-DD): 93
   • DD-MM-YYYY: 12
   • Nulas: 0

📝 Ejemplos de fechas en formato DD-MM-YYYY:
   • 02-07-2023
   • 12-03-2023
   • 23-12-2023


In [27]:
# =============================================================================
# 7. ANÁLISIS DE COLUMNA 'status' (ESTADO)
# =============================================================================
print("\n" + "=" * 70)
print("7. ANÁLISIS DE ESTADOS (status)")
print("=" * 70)

print(f"\n📊 Distribución de estados:")
status_counts = df['status'].value_counts()
for status, count in status_counts.items():
    status_display = "'VACÍO'" if status == '' else status
    print(f"   • {status_display}: {count}")

estados_vacios = (df['status'] == '').sum()
print(f"\n⚠️ Estados vacíos: {estados_vacios}")


7. ANÁLISIS DE ESTADOS (status)

📊 Distribución de estados:
   • CANCELADA: 22
   • FALLIDA: 21
   • COMPLETADA: 20
   • PENDIENTE: 17

⚠️ Estados vacíos: 0


In [28]:
# =============================================================================
# 8. ANÁLISIS DE COLUMNA 'store' (TIENDA)
# =============================================================================
print("\n" + "=" * 70)
print("8. ANÁLISIS DE TIENDAS (store)")
print("=" * 70)

print(f"\n🏪 Distribución de tiendas:")
store_counts = df['store'].value_counts(dropna=False)
for store, count in store_counts.items():
    store_display = 'NULO' if pd.isna(store) else store
    print(f"   • {store_display}: {count}")

tiendas_nulas = df['store'].isnull().sum()
print(f"\n⚠️ Tiendas nulas: {tiendas_nulas}")


8. ANÁLISIS DE TIENDAS (store)

🏪 Distribución de tiendas:
   • Tienda_Sur: 29
   • Tienda_Norte: 28
   • NULO: 19
   • Tienda_Este: 17
   • Tienda_Centro: 12

⚠️ Tiendas nulas: 19


In [29]:
# =============================================================================
# 9. ANÁLISIS DE COLUMNA 'customer_id' (CLIENTE)
# =============================================================================
print("\n" + "=" * 70)
print("9. ANÁLISIS DE CLIENTES (customer_id)")
print("=" * 70)

clientes_nulos = df['customer_id'].isnull().sum()
clientes_unicos = df['customer_id'].nunique()

print(f"\n👥 Estadísticas de clientes:")
print(f"   • Clientes únicos: {clientes_unicos}")
print(f"   • Clientes nulos: {clientes_nulos}")
print(f"   • Total transacciones: {len(df)}")



9. ANÁLISIS DE CLIENTES (customer_id)

👥 Estadísticas de clientes:
   • Clientes únicos: 42
   • Clientes nulos: 7
   • Total transacciones: 105


In [30]:
# =============================================================================
# 10. RESUMEN Y REPORTE
# =============================================================================
print("\n" + "=" * 70)
print("10. RESUMEN DE PROBLEMAS IDENTIFICADOS")
print("=" * 70)

resumen = {
    'fecha_analisis': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'dataset': {
        'nombre': 'transacciones_raw.csv',
        'filas': len(df),
        'columnas': len(df.columns)
    },
    'calidad_datos': {
        'duplicados_totales': int(duplicados),
        'duplicados_por_id': int(duplicados_por_id),
        'total_nulos': int(df.isnull().sum().sum()),
        'porcentaje_nulos_global': float((df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100)
    },
    'problemas_por_columna': {
        'transaction_id': {
            'duplicados': int(duplicados_por_id)
        },
        'customer_id': {
            'nulos': int(clientes_nulos),
            'porcentaje_nulos': float((clientes_nulos / len(df)) * 100)
        },
        'amount': {
            'nulos': int(montos_nulos),
            'negativos': int(montos_negativos),
            'formato_string': int(montos_string),
            'porcentaje_problemas': float(((montos_nulos + montos_negativos + montos_string) / len(df)) * 100)
        },
        'date': {
            'formato_no_iso': int(formato_ddmmyyyy),
            'nulos': int(fechas_nulas),
            'porcentaje_no_iso': float((formato_ddmmyyyy / len(df)) * 100)
        },
        'status': {
            'vacios': int(estados_vacios),
            'porcentaje_vacios': float((estados_vacios / len(df)) * 100)
        },
        'store': {
            'nulos': int(tiendas_nulas),
            'porcentaje_nulos': float((tiendas_nulas / len(df)) * 100)
        }
    }
}

# Guardar reporte JSON
with open('homeworks/output/reporte_calidad.json', 'w', encoding='utf-8') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

print("\n📁 Reporte guardado en: homeworks/output/reporte_calidad.json")


10. RESUMEN DE PROBLEMAS IDENTIFICADOS

📁 Reporte guardado en: homeworks/output/reporte_calidad.json


In [31]:
# =============================================================================
# 11. RECOMENDACIONES INICIALES
# =============================================================================
print("\n" + "=" * 70)
print("11. RECOMENDACIONES PARA LIMPIEZA")
print("=" * 70)

print("""
Basado en el análisis, se recomienda:

1. AMOUNT:
   ✓ Convertir strings con $ a números float
   ✓ Convertir valores negativos a positivos (o investigar causa)
   ✓ Decidir estrategia para nulos (0.0, media, o excluir)

2. DATE:
   ✓ Estandarizar todas las fechas al formato YYYY-MM-DD
   ✓ Validar fechas dentro del rango esperado (año 2023)

3. STATUS:
   ✓ Asignar valor por defecto para estados vacíos
   ✓ Validar valores permitidos: COMPLETADA, PENDIENTE, FALLIDA, CANCELADA

4. STORE:
   ✓ Asignar 'TIENDA_SIN_ESPECIFICAR' para valores nulos

5. CUSTOMER_ID:
   ✓ Asignar 'DESCONOCIDO' para valores nulos

6. DUPLICADOS:
   ✓ Eliminar registros duplicados basado en transaction_id
""")

print("\n" + "=" * 70)
print("✅ TAREA 1 COMPLETADA")
print("=" * 70)

# Verificar archivos generados
print("\n📁 Archivos generados:")
if os.path.exists('homeworks/output/reporte_calidad.json'):
    size = os.path.getsize('homeworks/output/reporte_calidad.json')
    print(f"   ✓ reporte_calidad.json ({size} bytes)")


11. RECOMENDACIONES PARA LIMPIEZA

Basado en el análisis, se recomienda:

1. AMOUNT:
   ✓ Convertir strings con $ a números float
   ✓ Convertir valores negativos a positivos (o investigar causa)
   ✓ Decidir estrategia para nulos (0.0, media, o excluir)

2. DATE:
   ✓ Estandarizar todas las fechas al formato YYYY-MM-DD
   ✓ Validar fechas dentro del rango esperado (año 2023)

3. STATUS:
   ✓ Asignar valor por defecto para estados vacíos
   ✓ Validar valores permitidos: COMPLETADA, PENDIENTE, FALLIDA, CANCELADA

4. STORE:
   ✓ Asignar 'TIENDA_SIN_ESPECIFICAR' para valores nulos

5. CUSTOMER_ID:
   ✓ Asignar 'DESCONOCIDO' para valores nulos

6. DUPLICADOS:
   ✓ Eliminar registros duplicados basado en transaction_id


✅ TAREA 1 COMPLETADA

📁 Archivos generados:
   ✓ reporte_calidad.json (882 bytes)
